# QLoRA Fine-Tuning: Linux CLI Assistant on TinyLlama 1.1B
**Stack:** TinyLlama-1.1B + 4-bit QLoRA (PEFT) + Hugging Face Transformers


In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.8 MB/s eta 0:00:00


In [13]:
# ── Cell 3: Load and format Hugging Face dataset ──────────────────────────────
from datasets import load_dataset

# 1. Download the first 1,000 examples using the active community mirror
print("Downloading CodeAlpaca dataset...")
dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train[:1000]") # <-- Updated Repo!

# 2. Format them to match your inference prompt exactly
def format_alpaca(example):
    instruction = example["instruction"]
    # If the example has extra context in the 'input' field, append it
    if example.get("input", "").strip():
        instruction += f"\n{example['input']}"

    prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n{example['output']}"
    )
    return {'text': prompt}

print("Formatting prompts...")
dataset = dataset.map(format_alpaca, remove_columns=dataset.column_names)

print(f"Dataset ready! Total examples: {len(dataset)}")
print("\nSample prompt:")
print(dataset[0]['text'][:300])

README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json:   0%|          | 0.00/8.06M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

Formatting prompts...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset ready! Total examples: 1000

Sample prompt:
### Instruction:
Create an array of length 5 which contains all even numbers between 1 and 10.

### Response:
arr = [2, 4, 6, 8, 10]


In [14]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={'': 0},
)

# --- THE NEW CRITICAL FIX ---
# This forces the Trainer to see the model as fp16, stopping it from
# reverting to bfloat16 during trainer.train()
model.config.torch_dtype = torch.float16
# ----------------------------

model.config.use_cache = False
model.config.pretraining_tp = 1

print(f'Model loaded. dtype: {model.dtype}')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded. dtype: torch.float16
GPU memory used: 1.86 GB


In [15]:
# ── Cell 5: Attach LoRA adapters ──────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                     # rank — higher = more capacity, more memory
    lora_alpha=32,            # scaling factor (keep 2x rank)
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

# --- THE FIX ---
# Hunt down any bfloat16 parameters that PEFT secretly initialized
# and convert them to float32 so the T4's GradScaler doesn't crash.
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)
# ---------------

model.print_trainable_parameters()
# Expect: ~0.4% of params trainable — that's the point of LoRA

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [16]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./tinyllama-linux-cli',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy='epoch',
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    report_to='none',
    dataset_text_field='text',
    max_length=512, # <-- Changed from max_seq_length to max_length
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,   # replaces 'tokenizer' in 0.9+
)

print('Trainer configured. Ready to train.')

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Trainer configured. Ready to train.


In [17]:
# ── Cell 7: Train ─────────────────────────────────────────────────────────────

# The ultimate safety net: Sweep for any rogue BFloat16 parameters
# that the Trainer secretly initialized and force them to float32
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

# Expected time on T4: ~15-25 min for 3 epochs over 68 examples
trainer.train()
print('Training complete!')

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.262042
20,0.937478
30,0.780051
40,0.738081
50,0.716306
60,0.698472
70,0.729156
80,0.660283
90,0.665731
100,0.698072


Training complete!


In [18]:
# ── Cell 8: Save adapter weights ──────────────────────────────────────────────
ADAPTER_DIR = './tinyllama-linux-cli-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to {ADAPTER_DIR}')

import os
files_saved = os.listdir(ADAPTER_DIR)
print(f'Files: {files_saved}')

Adapter saved to ./tinyllama-linux-cli-adapter
Files: ['tokenizer_config.json', 'tokenizer.json', 'chat_template.jinja', 'adapter_config.json', 'README.md', 'adapter_model.safetensors']


In [19]:
# ── Cell 9: Quick inference test ──────────────────────────────────────────────
import torch

model.gradient_checkpointing_disable()  # Turn off training memory tricks
model.config.use_cache = True           # Turn ON inference memory cache
model.eval()                            # Lock the weights (evaluation mode)
# ----------------------------------------------------

def ask(question, max_new_tokens=200):
    prompt = f'### Instruction:\n{question}\n\n### Response:\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split('### Response:')[-1].strip()

# Test it
test_questions = [
    'How do I delete a file in Linux?',
    'Write a Python function to calculate the Fibonacci sequence.',
    'What is the difference between a list and a dictionary in Python?',
]

for q in test_questions:
    print(f'Q: {q}')
    print(f'A: {ask(q)}')
    print('---')

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I delete a file in Linux?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: To delete a file in Linux, you need to use the command "rm". This command will remove the file from the file system.
---
Q: Write a Python function to calculate the Fibonacci sequence.


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: def fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci(n-1) + fibonacci(n-2)
---
Q: What is the difference between a list and a dictionary in Python?
A: A list is a collection of objects, while a dictionary is a collection of key-value pairs. A dictionary is a more powerful data structure than a list, as it allows for the storage of multiple values for each key.
---


In [20]:
# ── Cell 10: Download adapter weights ─────────────────────────────────────────
# Zip and download so you can use them for the merge+GGUF step
!zip -r tinyllama-linux-cli-adapter.zip tinyllama-linux-cli-adapter/

from google.colab import files
files.download('tinyllama-linux-cli-adapter.zip')
print('Download triggered. Check your browser downloads.')

  adding: tinyllama-linux-cli-adapter/ (stored 0%)
  adding: tinyllama-linux-cli-adapter/tokenizer_config.json (deflated 46%)
  adding: tinyllama-linux-cli-adapter/tokenizer.json (deflated 85%)
  adding: tinyllama-linux-cli-adapter/chat_template.jinja (deflated 60%)
  adding: tinyllama-linux-cli-adapter/adapter_config.json (deflated 59%)
  adding: tinyllama-linux-cli-adapter/README.md (deflated 65%)
  adding: tinyllama-linux-cli-adapter/adapter_model.safetensors (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered. Check your browser downloads.


In [2]:
# ── Restore Adapter Weights ───────────────────────────────────────────────────
import zipfile
import os
from google.colab import files

print("Please upload tinyllama-linux-cli-adapter.zip...")
uploaded = files.upload()

for file_name in uploaded.keys():
    print(f"Extracting {file_name}...")
    with zipfile.ZipFile(file_name, 'r') as zip_ref:
        zip_ref.extractall('.')

print("Adapter restored! Contents:")
print(os.listdir('./tinyllama-linux-cli-adapter'))

Please upload tinyllama-linux-cli-adapter.zip...


Saving tinyllama-linux-cli-adapter.zip to tinyllama-linux-cli-adapter.zip
Extracting tinyllama-linux-cli-adapter.zip...
Adapter restored! Contents:
['tokenizer.json', 'tokenizer_config.json', 'adapter_config.json', 'chat_template.jinja', 'adapter_model.safetensors', 'README.md']


In [3]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 12.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [4]:
# ── Cell 11: Merge LoRA Weights with Base Model ──────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_DIR = "./tinyllama-linux-cli-adapter"
MERGED_DIR = "./tinyllama-linux-cli-merged"

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16, # Load in FP16 for clean merging
    device_map="cpu"           # Merge on CPU to avoid running out of VRAM
)

print("Loading adapter weights...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

print("Merging weights permanently...")
merged_model = model.merge_and_unload()

print(f"Saving fully merged model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print("Merge complete!")

Loading base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading adapter weights...
Merging weights permanently...
Saving fully merged model to ./tinyllama-linux-cli-merged...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merge complete!


In [5]:
# ── Cell 12: Clone llama.cpp and install requirements ────────────────────────
!git clone https://github.com/ggerganov/llama.cpp
!pip install -r llama.cpp/requirements.txt
print("llama.cpp environment ready for conversion.")

Cloning into 'llama.cpp'...
remote: Enumerating objects: 98037, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 98037 (delta 107), reused 99 (delta 89), pack-reused 97854 (from 2)
Receiving objects: 100% (98037/98037), 395.95 MiB | 17.95 MiB/s, done.
Resolving deltas: 100% (69995/69995), done.
Updating files: 100% (2947/2947), done.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 937.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB

llama.cpp environment ready for conversion.


In [8]:
# ── Cell 13: Convert to GGUF and Quantize ────────────────────────────────────
# 1. Convert the merged model folder into a single unquantized GGUF file
# (This already succeeded, but it's safe to overwrite)
!python llama.cpp/convert_hf_to_gguf.py ./tinyllama-linux-cli-merged --outfile tinyllama-linux-cli.gguf

# 2. Compile the quantization tool using CMake (The NEW way)
!mkdir -p llama.cpp/build
!cmake -B llama.cpp/build llama.cpp
!cmake --build llama.cpp/build --config Release --target llama-quantize -j 4

# 3. Quantize the model down to 4-bit (Q4_K_M)
# Note the new path to the binary in the build/bin/ folder
!./llama.cpp/build/bin/llama-quantize tinyllama-linux-cli.gguf tinyllama-linux-cli-Q4_K_M.gguf Q4_K_M

print("\nQuantization complete! Checking final file sizes:")
!ls -lh tinyllama-linux-cli*.gguf

INFO:hf-to-gguf:Loading model: tinyllama-linux-cli-merged
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics detected float16 tensor dtype, setting --outtype f16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.w

In [9]:
# ── Cell 14: Download the final quantized GGUF ───────────────────────────────
from google.colab import files
files.download('tinyllama-linux-cli-Q4_K_M.gguf')
print('Download triggered! Transfer this file to your Jetson Nano.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered! Transfer this file to your Jetson Nano.
